In [1]:
# Downloading image datasets from kaggle
!kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images

Dataset URL: https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images
License(s): other




  0%|          | 0.00/105M [00:00<?, ?B/s]
  3%|▎         | 3.00M/105M [00:00<00:03, 31.1MB/s]
  8%|▊         | 8.00M/105M [00:00<00:03, 32.2MB/s]
 12%|█▏        | 13.0M/105M [00:00<00:02, 39.7MB/s]
 19%|█▉        | 20.0M/105M [00:00<00:02, 43.7MB/s]
 24%|██▍       | 25.0M/105M [00:00<00:02, 36.4MB/s]
 28%|██▊       | 29.0M/105M [00:00<00:02, 30.7MB/s]
 32%|███▏      | 33.0M/105M [00:01<00:03, 24.2MB/s]
 34%|███▍      | 36.0M/105M [00:01<00:03, 22.5MB/s]
 37%|███▋      | 39.0M/105M [00:01<00:02, 22.9MB/s]
 43%|████▎     | 45.0M/105M [00:01<00:02, 28.6MB/s]
 47%|████▋     | 49.0M/105M [00:01<00:01, 30.8MB/s]
 51%|█████     | 53.0M/105M [00:01<00:01, 31.6MB/s]
 55%|█████▍    | 57.0M/105M [00:02<00:01, 29.1MB/s]
 57%|█████▋    | 60.0M/105M [00:02<00:01, 27.7MB/s]
 60%|██████    | 63.0M/105M [00:02<00:01, 26.1MB/s]
 63%|██████▎   | 66.0M/105M [00:02<00:01, 25.8MB/s]
 66%|██████▌   | 69.0M/105M [00:02<00:01, 24.7MB/s]
 69%|██████▉   | 72.0M/105M [00:02<00:01, 23.9MB/s]
 72%|███████▏  | 75.

In [2]:
import os 
import zipfile

# downloading the data and making sure to not accidentally download it again
if not os.path.exists('../data/processed/cifake_data'):
    with zipfile.ZipFile('cifake-real-and-ai-generated-synthetic-images.zip', 'r') as zip_ref:
        zip_ref.extractall('../data/processed/cifake_data')
else:
    print("Data already downloaded.")

# Check if folders were created correctly
print(f"Data folders: {os.listdir('../data/processed/cifake_data')}")


Data already downloaded.


In [3]:
from torchvision import transforms

# defining the image transform to "scale" the images
transformer = transforms.Compose([
    # resizing for consistency
    transforms.Resize((32, 32)),

    # converting to a grid of numbers and normalizing them 
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

In [4]:
from torchvision import datasets
from torch.utils.data import DataLoader

# transforming images based on transformer above
def transform_images():
    dataset = datasets.ImageFolder(root='../data/processed/cifake_data/train', transform=transformer)
   
    # giving the model 100 images at a time (shuffled)
    loaded_data = DataLoader(dataset, batch_size=100, shuffle=True)
    return loaded_data

# Checking class labels (e.g., is AI 0 or 1?)
print(f"Total images: {len(dataset)}")
print(f"Class mapping: {dataset.class_to_idx}")


In [ ]:
import torch.nn as nn

# defining the neural network (Used Claude for specific numbers on pooling layers and linear layers. Prompt: help me create a model that ends with a linear layer so that the output is a number. note, explain each decision you make and why)
model = nn.Sequential(

    # finding patterns in the image
    nn.Conv2d(3, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),
    
    # finding more complex patterns
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),

    # flattening the 2D image into a 1D list
    nn.Flatten(),
    nn.Linear(32 * 8 * 8, 128),
    nn.ReLU(),

    # outputting a final score (real or fake)
    nn.Linear(128, 1)
)

In [6]:
import torch.optim as optim

#allowing the model to self-correct by calculating error and self adjusting
criterion = nn.BCEWithLogitsLoss() # calculates error
optimizer = optim.Adam(model.parameters(), lr=0.001) # makes adjustments

In [ ]:
def train_model(loaded_data):
    model.train()
    total_loss = 0

    for images, labels in loaded_data:
        # Checking the shape of the first batch
        if total_loss == 0:
            print(f"Image batch shape: {images.shape}")

        # reset previous adjustments (Claude. Prompt: how do i train this fresh)
        optimizer.zero_grad()

        guess = model(images).squeeze()

        # calculate how wrong the guess was
        #labels.float() so that it PyTorch doesn't throw an error
        loss = criterion(guess, labels.float())

        #adjusting
        loss.backward()
        optimizer.step()

        #summing up the loss
        total_loss += loss.item()
        
        # Print progress every 100 batches
        if len(loaded_data) > 100 and (total_loss / loss.item()) % 100 == 0:
            print(f"Batch {int(total_loss / loss.item())} processed...")

    # display error for epoch of training
    epoch_loss = total_loss / len(loaded_data)
    print(f"Epoch trained. Loss: {epoch_loss:.4f}\n")

In [ ]:
import torch

epochs = 5

# load our images ready for training
train_loader = transform_images()

# train the model for multiple epochs
for epoch in range(epochs):
    print(f"Training epoch [{epoch+1}/{epochs}]...")
    train_model(train_loader)

#save model when finished
torch.save(model.state_dict(), '../models/PicDetective_model.pth')
print("Model saved!")

Training epoch [1/5]...
Epoch trained. Loss: 0.3190

Training epoch [2/5]...
Epoch trained. Loss: 0.2093

Training epoch [3/5]...
Epoch trained. Loss: 0.1799

Training epoch [4/5]...
Epoch trained. Loss: 0.1611

Training epoch [5/5]...
Epoch trained. Loss: 0.1459

Model saved!


In [13]:
from torchvision import datasets
from torch.utils.data import DataLoader

# load test images
test_dataset = datasets.ImageFolder(root='../data/processed/cifake_data/test', transform=transformer)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

def evaluate_model(model, loader):
    model.eval()
    correct = 0
    total_correct = 0
    total = 0

    #evaluating while making sure not to change anything
    with torch.no_grad():
        for images, labels in loader:
            # get the model's raw score
            outputs = model(images).squeeze()

            # convert raw scores to probabilities and assinging a 0 or 1 
            probabilities = torch.sigmoid(outputs)
            predictions = (probabilities > 0.5).float()

            #totaling
            total += labels.size(0)

            # count correct guesses
            num_correct = (predictions == labels.float())
            total_correct += num_correct.sum().item()

    # print final accuracy
    accuracy = (total_correct / total) * 100
    print(f"Model Accuracy on Test Set: {accuracy:.2f}%")

In [14]:
import torch

#load last saved model and call evaluate_model function with it
model.load_state_dict(torch.load('../models/PicDetective_model.pth'))
evaluate_model(model, test_loader)

Model Accuracy on Test Set: 94.10%
